In [1]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from scipy.optimize import brentq
import matplotlib.pylab as plt
from scipy.optimize import least_squares

## Import Dataset

In [2]:
sheets = pd.ExcelFile('Swap and Swaption Markets_Amended.xlsx').sheet_names

In [3]:
df_swaption = pd.read_excel('Swap and Swaption Markets_Amended.xlsx', sheet_name=sheets[3], header = 2)
df_swaption

,Expiry,Tenor,-200bps,-150bps,-100bps,-50bps,-25bps,ATM,+25bps,+50bps,+100bps,+150bps,+200bps
0,1Y,1Y,91.570,62.030,44.130,31.224,26.182,22.50,20.96,21.40,24.34,27.488,30.297
1,1Y,2Y,83.270,61.240,46.570,35.807,31.712,28.72,27.12,26.84,28.51,31.025,33.523
2,1Y,3Y,73.920,56.870,44.770,35.745,32.317,29.78,28.29,27.80,28.77,30.725,32.833
3,1Y,5Y,55.190,44.640,36.510,30.242,27.851,26.07,24.98,24.56,25.12,26.536,28.165
4,1Y,10Y,41.180,35.040,30.207,26.619,25.351,24.47,23.98,23.82,24.25,25.204,26.355
5,5Y,1Y,67.800,49.090,38.400,31.485,29.060,27.26,26.04,25.32,24.94,25.320,25.980
6,5Y,2Y,57.880,46.410,39.033,33.653,31.531,29.83,28.56,27.65,26.71,26.540,26.760
7,5Y,3Y,53.430,44.440,38.180,33.437,31.536,29.98,28.76,27.82,26.67,26.200,26.150
8,5Y,5Y,41.990,36.524,32.326,29.005,27.677,26.60,25.73,25.02,24.06,23.570,23.400
9,5Y,10Y,34.417,30.948,28.148,25.954,25.136,24.51,23.99,23.56,22.91,22.490,22.250


Notes on Swaption Data on first inspection:
* This is lognormal Implied Vol for SOFR Swaptions
* Convection for bps strike is (Forward + Basis Point)

In [4]:
def convert_tenor_to_numeric(series):
    num = series.str.slice(stop = -1).astype(int)
    freq = series.str.slice(start = -1)

    conditions = [freq.str.upper() == 'M', freq.str.upper() == 'Y']
    choices = [num * 30 / 360, num.astype(float)]

    return pd.Series(np.select(conditions, choices, default=np.nan), name=series.name)

In [5]:
def convert_bps_to_numeric(column_series):

    column_names = column_series.str.slice(stop = -3)
    replaced_list = []

    for col in column_names:
        if col == '':
            replaced_list.append(0.0)
        else:
            replaced_list.append(float(col) / 10000)
    
    return replaced_list  

In [6]:
# Extract the expiry and tenor information from the column names and convert them to numeric values
# Imp vol is calculated in percentages, so we need to divide by 100 to get the decimal form
# bps is defined as 1/100 of a percentage point, so we need to divide by 10000 to get the decimal form

df_timings = pd.concat([convert_tenor_to_numeric(df_swaption['Expiry']), convert_tenor_to_numeric(df_swaption['Tenor'])], axis=1)
df_bps = df_swaption.drop(columns = ['Expiry', 'Tenor']) / 100  # e.g. 22.5 -> 0.225
df_bps.columns = convert_bps_to_numeric(df_bps.columns)

display(pd.concat([df_timings, df_bps], axis=1))

,Expiry,Tenor,-0.02,-0.015,-0.01,-0.005,-0.0025,0.0,0.0025,0.005,0.01,0.015,0.02
0,1.0,1.0,0.91570,0.62030,0.44130,0.31224,0.26182,0.2250,0.2096,0.2140,0.2434,0.27488,0.30297
1,1.0,2.0,0.83270,0.61240,0.46570,0.35807,0.31712,0.2872,0.2712,0.2684,0.2851,0.31025,0.33523
2,1.0,3.0,0.73920,0.56870,0.44770,0.35745,0.32317,0.2978,0.2829,0.2780,0.2877,0.30725,0.32833
3,1.0,5.0,0.55190,0.44640,0.36510,0.30242,0.27851,0.2607,0.2498,0.2456,0.2512,0.26536,0.28165
4,1.0,10.0,0.41180,0.35040,0.30207,0.26619,0.25351,0.2447,0.2398,0.2382,0.2425,0.25204,0.26355
5,5.0,1.0,0.67800,0.49090,0.38400,0.31485,0.29060,0.2726,0.2604,0.2532,0.2494,0.25320,0.25980
6,5.0,2.0,0.57880,0.46410,0.39033,0.33653,0.31531,0.2983,0.2856,0.2765,0.2671,0.26540,0.26760
7,5.0,3.0,0.53430,0.44440,0.38180,0.33437,0.31536,0.2998,0.2876,0.2782,0.2667,0.26200,0.26150
8,5.0,5.0,0.41990,0.36524,0.32326,0.29005,0.27677,0.2660,0.2573,0.2502,0.2406,0.23570,0.23400
9,5.0,10.0,0.34417,0.30948,0.28148,0.25954,0.25136,0.2451,0.2399,0.2356,0.2291,0.22490,0.22250


We can also output the actual prices of the swaptions using the Black Scholes Lognormal model, which is the same as the Black76 model for swaptions.

In [7]:
## Default Functions for Black Scholes Lognormal Model. 
## In the event of a Black76 model, set S = F and r = 0.0.
## If there is dividend, set r = r-q where q is the dividend yield.

def BlackScholesLognormalPayer(S, K, r, sigma, T):
    d1 = (np.log(S/K)+(r+sigma**2/2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)


def BlackScholesLognormalReceiver(S, K, r, sigma, T):
    d1 = (np.log(S/K)+(r+sigma**2/2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return K*np.exp(-r*T)*norm.cdf(-d2) - S*norm.cdf(-d1)

In [8]:
# Build combined surface
df_surface = pd.concat([df_timings, df_bps], axis=1)

In [11]:
def calculate_forward_rate(T, tenor, method='tenor_based'):
    """
    Calculate forward swap rate for a given expiry and tenor.
    
    Parameters:
    -----------
    T : float
        Time to expiry (in years)
    tenor : float
        Swap tenor (in years)
    method : str
        'tenor_based': Simple approximate forward rate based on tenor
        'flat': Assumes flat yield curve
        
    Returns:
    --------
    float : Forward swap rate
    """
    if method == 'tenor_based':
        # Estimated forward rate based on tenor (longer tenors tend to have higher rates)
        # This is a simplified assumption typical in swaps market
        base_rate = 0.02  # 2% base rate
        tenor_adjustment = 0.003 * np.log(tenor + 1)  # Small positive adjustment for longer tenors
        forward_rate = base_rate + tenor_adjustment
    else:  # flat
        forward_rate = 0.02
    
    return forward_rate


def calculate_annuity_factor(T, tenor, rate=0.02, frequency=4):
    """
    Calculate annuity/PVBP for a swap expiring at time T with given tenor.
    
    Parameters:
    -----------
    T : float
        Time to expiry (in years)
    tenor : float
        Swap tenor (in years)
    rate : float
        Discount rate (assumed constant for flat yield curve)
    frequency : int
        Payment frequency per year (4 = quarterly, 2 = semi-annual)
        
    Returns:
    --------
    float : Annuity factor (PVBP) - present value of 1bp per period
    """
    # Number of periods in the swap tenor
    num_periods = int(tenor * frequency)
    
    # Annuity factor = sum of discount factors for each period
    annuity = 0.0
    period_length = 1.0 / frequency
    
    for i in range(1, num_periods + 1):
        time_to_period = T + i * period_length
        discount_factor = np.exp(-rate * time_to_period)
        annuity += discount_factor * period_length
    
    return annuity


def calculate_prices(df_bps, df_timings, compute_forward_and_annuity=True):
    """
    Calculate swaption prices from implied volatilities.
    
    Parameters:
    -----------
    df_bps : DataFrame
        Implied volatilities indexed by expiry, columns are strike shifts
    df_timings : DataFrame
        Contains 'Expiry' and 'Tenor' columns with timing information
    compute_forward_and_annuity : bool
        If True, calculate forward rates and annuities; if False, use normalized values
        
    Returns:
    --------
    payer_prices : DataFrame
        Payer swaption prices
    receiver_prices : DataFrame
        Receiver swaption prices
    df_swaption_prices : DataFrame
        Combined dataframe with timings and all prices
    df_forwards : DataFrame
        Forward swap rates for each expiry-tenor
    df_annuities : DataFrame
        Annuity factors for each expiry-tenor
    """
    r = 0.0  # Black76 discounting convention
    strike_shifts = df_bps.columns.astype(float).to_list()
    T_vec = df_timings['Expiry'].astype(float).values
    tenors = df_timings['Tenor'].astype(float).values
    
    # Initialize lists to store forward rates and annuities
    forward_rates = []
    annuity_factors = []
    
    payer_strike_shifts = [shift for shift in strike_shifts if shift >= 0]
    receiver_strike_shifts = [shift for shift in strike_shifts if shift < 0]

    payer_prices = pd.DataFrame(index=df_bps.index, columns=strike_shifts, dtype=float)
    receiver_prices = pd.DataFrame(index=df_bps.index, columns=strike_shifts, dtype=float)

    for idx, (T, tenor) in enumerate(zip(T_vec, tenors)):
        # Calculate forward rate and annuity for this expiry-tenor pair
        if compute_forward_and_annuity:
            F = calculate_forward_rate(T, tenor)
            annuity = calculate_annuity_factor(T, tenor)
        else:
            F = 1.0  # normalized
            annuity = 1.0  # normalized
        
        forward_rates.append(F)
        annuity_factors.append(annuity)

        for shift in strike_shifts:
            K = F + shift
            vol = df_bps.iloc[idx][shift]

            if not np.isnan(vol):
                payer_prices.iloc[idx, payer_prices.columns.get_loc(shift)] = (
                    annuity * BlackScholesLognormalPayer(F, K, r, vol, T)
                )
                receiver_prices.iloc[idx, receiver_prices.columns.get_loc(shift)] = (
                    annuity * BlackScholesLognormalReceiver(F, K, r, vol, T)
                )

    payer_prices = payer_prices.drop(columns=receiver_strike_shifts)
    receiver_prices = receiver_prices.drop(columns=payer_strike_shifts)

    # Create dataframes for forward rates and annuities
    df_forwards = df_timings.copy()
    df_forwards['Forward_Rate'] = forward_rates
    
    df_annuities = df_timings.copy()
    df_annuities['Annuity'] = annuity_factors

    return (payer_prices, receiver_prices, 
            pd.concat([df_timings, receiver_prices, payer_prices], axis=1),
            df_forwards, df_annuities)

In [12]:
# Attach expiry/tenor for readability and calculate forward rates and annuities
payer_swaption_prices, receiver_swaption_prices, df_swaption_prices, df_forwards, df_annuities = calculate_prices(df_bps, df_timings, compute_forward_and_annuity=True)

print("=" * 80)
print("SWAPTION PRICES (PAYER AND RECEIVER)")
print("=" * 80)
display(df_swaption_prices)

print("\n" + "=" * 80)
print("FORWARD SWAP RATES BY EXPIRY AND TENOR")
print("=" * 80)
display(df_forwards.round(6))

print("\n" + "=" * 80)
print("ANNUITY FACTORS (PVBP) BY EXPIRY AND TENOR")
print("=" * 80)
display(df_annuities.round(6))

SWAPTION PRICES (PAYER AND RECEIVER)


,Expiry,Tenor,-0.02,-0.015,-0.01,-0.005,-0.0025,0.0,0.0025,0.005,0.01,0.015,0.02
0,1.0,1.0,0.000009,0.000095,0.000270,0.000673,0.001106,0.001915,0.000915,0.000459,0.000169,0.000087,0.000053
1,1.0,2.0,0.000041,0.000298,0.000858,0.002094,0.003262,0.005099,0.003031,0.001801,0.000760,0.000407,0.000255
2,1.0,3.0,0.000058,0.000422,0.001318,0.003379,0.005283,0.008141,0.005063,0.003105,0.001300,0.000662,0.000393
3,1.0,5.0,0.000021,0.000278,0.001292,0.004347,0.007461,0.012243,0.007352,0.004271,0.001511,0.000634,0.000315
4,1.0,10.0,0.000008,0.000211,0.001616,0.007390,0.013760,0.023467,0.014570,0.008759,0.003153,0.001238,0.000550
5,5.0,1.0,0.000191,0.000866,0.001693,0.002882,0.003703,0.004725,0.003737,0.002967,0.001951,0.001378,0.001035
6,5.0,2.0,0.000493,0.001973,0.004059,0.006891,0.008671,0.010769,0.008802,0.007183,0.004863,0.003435,0.002542
7,5.0,3.0,0.000864,0.003135,0.006386,0.010772,0.013498,0.016668,0.013746,0.011290,0.007651,0.005315,0.003831
8,5.0,5.0,0.000889,0.003735,0.008544,0.015604,0.020151,0.025483,0.020854,0.016950,0.011126,0.007373,0.005008
9,5.0,10.0,0.001422,0.006041,0.014735,0.028379,0.037415,0.048037,0.039623,0.032519,0.021683,0.014409,0.009649



FORWARD SWAP RATES BY EXPIRY AND TENOR


,Expiry,Tenor,Forward_Rate
0,1.0,1.0,0.022079
1,1.0,2.0,0.023296
2,1.0,3.0,0.024159
3,1.0,5.0,0.025375
4,1.0,10.0,0.027194
5,5.0,1.0,0.022079
6,5.0,2.0,0.023296
7,5.0,3.0,0.024159
8,5.0,5.0,0.025375
9,5.0,10.0,0.027194



ANNUITY FACTORS (PVBP) BY EXPIRY AND TENOR


,Expiry,Tenor,Annuity
0,1.0,1.0,0.968038
1,1.0,2.0,1.916907
2,1.0,3.0,2.846987
3,1.0,5.0,4.652262
4,1.0,10.0,8.861802
5,5.0,1.0,0.893611
6,5.0,2.0,1.769528
7,5.0,3.0,2.628100
8,5.0,5.0,4.294579
9,5.0,10.0,8.180475


There are two ways to calibrate the model:

1. Calibrate the model to the prices derived from the implied vol reporting models - i.e.: Output Black from implied vol to price, then use calibrate pricing model based on the outputted prices
2. Calibrate the model based on the implied vol reporting models directly

# Part 2: Swaption Calibration

### A: Displaced-Diffusion Model 

This model can be built on top of the Black lognormal Model

$$
V_{n,N}(0) = P_{n+1, N}(0) \cdot Black\left(\frac{S_{n,N}(0)}{\beta}, K+\frac{1-\beta}{\beta}S_{n,N}(0), \sigma\beta, T \right)
$$

Given the above, $\sigma_{dd} = \beta\times\sigma_{black}$. This can be optimised 

In [13]:
# def calibrate_displaced_diffusion(df_timings, df_payer_prices, df_receiver_prices):

#     annuity = 1.0
#     forward_rate = 1.0

#     for row in df_timings.itertuples():
        
#         i = row.Index



In [14]:
# calibrate_displaced_diffusion(df_timings, payer_swaption_prices, receiver_swaption_prices)

In [15]:

# Displaced Diffusion (Shifted Lognormal) Model Calibration
# Formula: Price = Black(F/beta, K + (1-beta)/beta * F, sigma*beta, T)

def DisplacedDiffusion_price(F, K, T, beta, sigma):
    """
    Calculate price under displaced diffusion model using Black76 formula.
    
    The displaced diffusion model adjusts the forward and strike:
    - Effective Forward: F_eff = F / beta
    - Effective Strike: K_eff = K + (1-beta)/beta * F
    - Effective Volatility: sigma_eff = sigma * beta
    
    Then applies Black's formula.
    """
    if beta <= 0 or beta > 1.0:
        return np.nan
    
    F_eff = F / beta
    K_eff = K + (1 - beta) / beta * F
    sigma_eff = sigma * beta
    
    # Use Black76 formula (put/call doesn't matter for our purposes, using payer convention)
    r = 0.0  # Black76 convention
    price = BlackScholesLognormalPayer(F_eff, K_eff, r, sigma_eff, T)
    
    return price


def DisplacedDiffusion_implied_vol(F, K, T, beta, sigma):
    """
    Calculate implied Black volatility for displaced diffusion model.
    Inverts the price to find what Black vol would produce the same price.
    """
    try:
        # Get price from DD model
        dd_price = DisplacedDiffusion_price(F, K, T, beta, sigma)
        
        if np.isnan(dd_price) or dd_price <= 0:
            return np.nan
        
        # Invert to get implied vol using brentq
        implied_vol = brentq(
            lambda vol: BlackScholesLognormalPayer(F, K, 0.0, vol, T) - dd_price,
            1e-12, 10.0
        )
        return implied_vol
    except:
        return np.nan


def dd_calibration_error(params, strikes, market_vols, F, T):
    """
    Calculate total squared error between market vols and displaced diffusion model vols.
    params = [beta, sigma]
    """
    beta, sigma = params
    
    # Validate bounds
    if beta <= 0 or beta > 1.0 or sigma <= 0:
        return 1e10
    
    error = 0.0
    for strike, market_vol in zip(strikes, market_vols):
        try:
            dd_vol = DisplacedDiffusion_implied_vol(F, strike, T, beta, sigma)
            if np.isnan(dd_vol):
                return 1e10
            error += (market_vol - dd_vol) ** 2
        except:
            return 1e10
    
    return error


def calibrate_beta_via_brentq(sigma, strikes, market_vols, F, T):
    """Use brentq to find beta that minimizes DD model error"""
    def error_func(beta):
        return dd_calibration_error([beta, sigma], strikes, market_vols, F, T)
    
    try:
        beta_opt = brentq(
            lambda b: error_func(b) - error_func(0.5),
            0.01, 0.99
        )
    except:
        beta_opt = 0.5
    
    return beta_opt


def calibrate_sigma_via_brentq(beta, strikes, market_vols, F, T):
    """Use brentq to find sigma that minimizes DD model error"""
    def error_func(sigma):
        return dd_calibration_error([beta, sigma], strikes, market_vols, F, T)
    
    try:
        sigma_opt = brentq(
            lambda s: error_func(s) - error_func(0.3),
            0.001, 2.0
        )
    except:
        sigma_opt = 0.3
    
    return sigma_opt


def calibrate_dd_parameters(strikes, market_vols, F, T, max_iterations=10):
    """Iterative calibration of beta and sigma using brentq"""
    beta_init, sigma_init = 0.5, 0.3
    
    for iteration in range(max_iterations):
        # Optimize each parameter sequentially
        beta_opt = calibrate_beta_via_brentq(sigma_init, strikes, market_vols, F, T)
        sigma_opt = calibrate_sigma_via_brentq(beta_opt, strikes, market_vols, F, T)
        
        # Check convergence
        conv_err = abs(beta_opt - beta_init) + abs(sigma_opt - sigma_init)
        
        if conv_err < 1e-6:
            break
        
        beta_init, sigma_init = beta_opt, sigma_opt
    
    return beta_opt, sigma_opt


# Extract setup parameters
F = 1.0
strikes_list = df_bps.columns.astype(float).to_list()

# Store results for displaced diffusion
dd_calibration_results = []

print("\n" + "=" * 80)
print("DISPLACED DIFFUSION (SHIFTED LOGNORMAL) CALIBRATION")
print("=" * 80)

# Iterate through each expiry-tenor combination
for expiry_idx in range(len(df_timings)):
    T = df_timings.iloc[expiry_idx, 0]  # Time to expiry
    tenor = df_timings.iloc[expiry_idx, 1]  # Tenor
    
    # Extract market implied vols for this expiry
    market_vols = df_bps.iloc[expiry_idx].values
    
    # Create arrays without NaN values
    valid_idx = ~np.isnan(market_vols)
    strikes = np.array(strikes_list)[valid_idx] + F
    market_vols_clean = market_vols[valid_idx]
    
    if len(strikes) > 0:
        print(f"\nExpiry: {T:.4f}Y, Tenor: {tenor:.4f}Y")
        print(f"  Calibrating to {len(strikes)} strikes...")
        
        try:
            # Calibrate parameters
            beta_calib, sigma_calib = calibrate_dd_parameters(
                strikes, market_vols_clean, F, T
            )
            
            # Calculate MSE
            mse_dd = dd_calibration_error(
                [beta_calib, sigma_calib], strikes, market_vols_clean, F, T
            ) / len(strikes)
            
            print(f"  beta={beta_calib:.6f}, sigma={sigma_calib:.6f}, MSE={mse_dd:.2e}")
            
            # Store result
            dd_calibration_results.append({
                'Expiry (Years)': T,
                'Tenor (Years)': tenor,
                'Num Strikes': len(strikes),
                'Beta': beta_calib,
                'Sigma': sigma_calib,
                'MSE': mse_dd
            })
        except Exception as e:
            print(f"  Calibration failed: {str(e)}")
    else:
        print(f"\nExpiry: {T:.4f}Y, Tenor: {tenor:.4f}Y - No valid data")

# Create results dataframe
df_dd_calibration = pd.DataFrame(dd_calibration_results)

print("\n" + "=" * 80)
print("\nDISPLACED DIFFUSION PARAMETERS - SUMMARY TABLE")
print("=" * 80)
display(df_dd_calibration.round(6))



DISPLACED DIFFUSION (SHIFTED LOGNORMAL) CALIBRATION

Expiry: 1.0000Y, Tenor: 1.0000Y
  Calibrating to 11 strikes...
  beta=0.500000, sigma=0.300000, MSE=4.76e-02

Expiry: 1.0000Y, Tenor: 2.0000Y
  Calibrating to 11 strikes...
  beta=0.500000, sigma=0.300000, MSE=3.74e-02

Expiry: 1.0000Y, Tenor: 3.0000Y
  Calibrating to 11 strikes...
  beta=0.500000, sigma=0.300000, MSE=2.63e-02

Expiry: 1.0000Y, Tenor: 5.0000Y
  Calibrating to 11 strikes...
  beta=0.500000, sigma=0.300000, MSE=8.98e-03

Expiry: 1.0000Y, Tenor: 10.0000Y
  Calibrating to 11 strikes...
  beta=0.500000, sigma=0.300000, MSE=3.22e-03

Expiry: 5.0000Y, Tenor: 1.0000Y
  Calibrating to 11 strikes...
  beta=0.500000, sigma=0.300000, MSE=1.75e-02

Expiry: 5.0000Y, Tenor: 2.0000Y
  Calibrating to 11 strikes...
  beta=0.500000, sigma=0.300000, MSE=1.03e-02

Expiry: 5.0000Y, Tenor: 3.0000Y
  Calibrating to 11 strikes...
  beta=0.500000, sigma=0.300000, MSE=7.64e-03

Expiry: 5.0000Y, Tenor: 5.0000Y
  Calibrating to 11 strikes...
  

,Expiry (Years),Tenor (Years),Num Strikes,Beta,Sigma,MSE
0,1.0,1.0,11,0.5,0.3,0.047637
1,1.0,2.0,11,0.5,0.3,0.037427
2,1.0,3.0,11,0.5,0.3,0.026255
3,1.0,5.0,11,0.5,0.3,0.008977
4,1.0,10.0,11,0.5,0.3,0.003219
5,5.0,1.0,11,0.5,0.3,0.017459
6,5.0,2.0,11,0.5,0.3,0.010280
7,5.0,3.0,11,0.5,0.3,0.007636
8,5.0,5.0,11,0.5,0.3,0.003425
9,5.0,10.0,11,0.5,0.3,0.003397


### B: SABR Model

The implied Black volatility of the SABR model is given below, where $\beta = 0.75$ as a default setting

$$
    \begin{split}
      \sigma_{SABR}(F_0, K, \alpha, \beta, \rho, \nu)
      = \frac{\alpha}{(F_0K)^{(1-\beta)/2}\left\{ 1 + \frac{(1-\beta)^2}{24}\log^2\left(\frac{F_0}{K}\right) + \frac{(1-\beta)^4}{1920}\log^4\left(\frac{F_0}{K}\right) + \cdots\right\} }
      \times \frac{z}{x(z)} \times \left\{ 1 + \left[
           \frac{(1-\beta)^2}{24}
           \frac{\alpha^2}{(F_0K)^{1-\beta}}+\frac{1}{4}\frac{\rho\beta\nu\alpha}{(F_0K)^{(1-\beta)/2}}+\frac{2-3\rho^2}{24}\nu^2\right]
         T + \cdots \right.
    \end{split}
$$

where

$$
    \begin{split}
      z = \frac{\nu}{\alpha} (F_0K)^{(1-\beta)/2}
      \log\left(\frac{F_0}{K}\right),
    \end{split}
$$

and

$$
    % \begin{split}
      x(z) = \log \left[ \frac{\sqrt{1-2\rho z+z^2}+z -\rho}{1-\rho}
      \right].
    % \end{split}
$$

In [16]:
def SABR(F, K, T, alpha, beta, rho, nu):
    X = K
    # if K is at-the-money-forward
    if abs(F - K) < 1e-12:
        numer1 = (((1 - beta)**2)/24)*alpha*alpha/(F**(2 - 2*beta))
        numer2 = 0.25*rho*beta*nu*alpha/(F**(1 - beta))
        numer3 = ((2 - 3*rho*rho)/24)*nu*nu
        VolAtm = alpha*(1 + (numer1 + numer2 + numer3)*T)/(F**(1-beta))
        sabrsigma = VolAtm
    else:
        z = (nu/alpha)*((F*X)**(0.5*(1-beta)))*np.log(F/X)
        zhi = np.log((((1 - 2*rho*z + z*z)**0.5) + z - rho)/(1 - rho))
        numer1 = (((1 - beta)**2)/24)*((alpha*alpha)/((F*X)**(1 - beta)))
        numer2 = 0.25*rho*beta*nu*alpha/((F*X)**((1 - beta)/2))
        numer3 = ((2 - 3*rho*rho)/24)*nu*nu
        numer = alpha*(1 + (numer1 + numer2 + numer3)*T)*z
        denom1 = ((1 - beta)**2/24)*(np.log(F/X))**2
        denom2 = (((1 - beta)**4)/1920)*((np.log(F/X))**4)
        denom = ((F*X)**((1 - beta)/2))*(1 + denom1 + denom2)*zhi
        sabrsigma = numer/denom

    return sabrsigma

The definition above contains the function

$$
\begin{split}
{SABR}(F, K, T, \alpha, \beta, \rho, \nu)
\end{split}
$$

The function returns a volatility $\sigma_{{SABR}}$ for the Black76Lognormal call or put option formula, so that

$$
\begin{split}
{Call price} &= {BlackScholesCall}(S, K, r, \sigma_{{SABR}}, T) \\
{Put price} &= {BlackScholesPut}(S, K, r, \sigma_{{SABR}}, T) \\
\end{split}
$$

How do we determine the parameters $\alpha$, $\rho$ and $\nu$?
- We choose them so that the output of the SABR model matches the implied volatilities observed in the market.
- We refer to this process as "model calibration".

In other words, defining

  $$
    \begin{split}
      \sigma_{{Mkt}}(K_1) - {SABR}(F, K_1, T, \alpha, 0.8, \rho, \nu) &= \epsilon_1 \\
      \sigma_{{Mkt}}(K_2) - {SABR}(F, K_2, T, \alpha, 0.8, \rho, \nu) &= \epsilon_2 \\
      \vdots&\\
      \sigma_{{Mkt}}(K_n) - {SABR}(F, K_n, T, \alpha, 0.8, \rho, \nu) &= \epsilon_n \\
    \end{split}
  $$

We want to minimize the sum of squared error terms as follows:
  
  $$
    \begin{split}
      \min_{\substack{\alpha,\; \rho,\; \nu}} \;\sum_{i=1}^n \epsilon_i^2
    \end{split}
  $$

We use the "least_squares" algorithm in "scipy" package to calibrate the SABR model parameters:


In [17]:
beta = 0.75

def sabrcalibration(x, strikes, vols, F, T):
    err = 0.0
    for i, vol in enumerate(vols):
        err += (vol - SABR(F, strikes[i], T,
                           x[0], beta, x[1], x[2]))**2

    return err


def impliedVolatility(S, K, r, price, T, payoff):
    try:
        if (payoff.lower() == 'call'):
            impliedVol = brentq(lambda x: price -
                                BlackScholesLognormalPayer(S, K, r, x, T),
                                1e-12, 10.0)
        elif (payoff.lower() == 'put'):
            impliedVol = brentq(lambda x: price -
                                BlackScholesLognormalReceiver(S, K, r, x, T),
                                1e-12, 10.0)
        else:
            raise NameError('Payoff type not recognized')
    except Exception:
        impliedVol = np.nan

    return impliedVol

In [21]:
# SABR Calibration from df_swaption prices using brentq - All expiry/tenor combinations

def total_squared_error(params, strikes, vols, F, T, beta):
    """Calculate total squared error between market vols and SABR vols"""
    alpha, rho, nu = params
    
    # Validate parameter bounds
    if alpha <= 0 or abs(rho) >= 1 or nu <= 0:
        return 1e10
    
    error = 0.0
    for strike, vol in zip(strikes, vols):
        try:
            sabr_vol = SABR(F, strike, T, alpha, beta, rho, nu)
            error += (vol - sabr_vol) ** 2
        except:
            return 1e10
    
    return error


def calibrate_alpha_via_brentq(rho, nu, strikes, vols, F, T, beta):
    """Use brentq to find alpha that minimizes total error"""
    def error_func(alpha):
        return total_squared_error([alpha, rho, nu], strikes, vols, F, T, beta)
    
    try:
        alpha_opt = brentq(lambda a: error_func(a) - error_func(0.01), 
                          0.001, 0.5)
    except:
        alpha_opt = 0.01
    
    return alpha_opt


def calibrate_rho_via_brentq(alpha, nu, strikes, vols, F, T, beta):
    """Use brentq to find rho that minimizes total error"""
    def error_func(rho):
        if abs(rho) >= 0.99:
            return 1e10
        return total_squared_error([alpha, rho, nu], strikes, vols, F, T, beta)
    
    try:
        rho_opt = brentq(lambda r: error_func(r) - error_func(0.0), 
                        -0.99, 0.99)
    except:
        rho_opt = 0.0
    
    return rho_opt


def calibrate_nu_via_brentq(alpha, rho, strikes, vols, F, T, beta):
    """Use brentq to find nu that minimizes total error"""
    def error_func(nu):
        if nu <= 0:
            return 1e10
        return total_squared_error([alpha, rho, nu], strikes, vols, F, T, beta)
    
    try:
        nu_opt = brentq(lambda n: error_func(n) - error_func(0.5), 
                       0.01, 2.0)
    except:
        nu_opt = 0.5
    
    return nu_opt


def calibrate_sabr_parameters(strikes, market_vols, F, T, beta, max_iterations=5):
    """Iterative calibration of alpha, rho, nu using brentq"""
    alpha_init, rho_init, nu_init = 0.02, 0.2, 0.3
    
    for iteration in range(max_iterations):
        # Optimize each parameter sequentially
        alpha_opt = calibrate_alpha_via_brentq(rho_init, nu_init, strikes, market_vols, F, T, beta)
        rho_opt = calibrate_rho_via_brentq(alpha_opt, nu_init, strikes, market_vols, F, T, beta)
        nu_opt = calibrate_nu_via_brentq(alpha_opt, rho_opt, strikes, market_vols, F, T, beta)
        
        # Check convergence
        conv_err = abs(alpha_opt - alpha_init) + abs(rho_opt - rho_init) + abs(nu_opt - nu_init)
        
        if conv_err < 1e-6:
            break
        
        alpha_init, rho_init, nu_init = alpha_opt, rho_opt, nu_opt
    
    return alpha_opt, rho_opt, nu_opt


# Extract strikes and forward rate
F = 1.0  # Normalized forward rate. Need to update if we want to use actual forward rates from market data
strikes_list = df_bps.columns.astype(float).to_list()
beta = 0.75

# Store results
calibration_results = []

print("SABR Calibration for all expiry-tenor combinations")
print("=" * 80)

# Iterate through each expiry
for expiry_idx in range(len(df_timings)):
    T = df_timings.iloc[expiry_idx, 0]  # Time to expiry
    tenor = df_timings.iloc[expiry_idx, 1]  # Tenor
    
    # Extract market implied vols for this expiry
    market_vols = df_bps.iloc[expiry_idx].values
    
    # Create arrays without NaN values
    valid_idx = ~np.isnan(market_vols)
    strikes = np.array(strikes_list)[valid_idx] + F
    market_vols_clean = market_vols[valid_idx]
    
    if len(strikes) > 0:
        print(f"\nExpiry: {T:.4f}Y, Tenor: {tenor:.4f}Y")
        print(f"  Calibrating to {len(strikes)} strikes...")
        
        try:
            # Calibrate parameters
            alpha_calib, rho_calib, nu_calib = calibrate_sabr_parameters(
                strikes, market_vols_clean, F, T, beta
            )
            
            # Calculate MSE
            mse = total_squared_error([alpha_calib, rho_calib, nu_calib], 
                                     strikes, market_vols_clean, F, T, beta) / len(strikes)
            
            print(f"  alpha={alpha_calib:.6f}, rho={rho_calib:.6f}, nu={nu_calib:.6f}, MSE={mse:.2e}")
            
            # Store result
            calibration_results.append({
                'Expiry (Years)': T,
                'Tenor (Years)': tenor,
                'Num Strikes': len(strikes),
                'Alpha': alpha_calib,
                'Beta': beta,
                'Rho': rho_calib,
                'Nu': nu_calib,
                'MSE': mse
            })
        except Exception as e:
            print(f"  Calibration failed: {str(e)}")
    else:
        print(f"\nExpiry: {T:.4f}Y, Tenor: {tenor:.4f}Y - No valid data")

# Create results dataframe
df_calibration = pd.DataFrame(calibration_results)

print("\n" + "=" * 80)
print("\nCALIBRATED SABR PARAMETERS - SUMMARY TABLE")
print("=" * 80)
display(df_calibration.round(6))

# Optional: Save to CSV
# df_calibration.to_csv('SABR_Calibration_Results.csv', index=False)


SABR Calibration for all expiry-tenor combinations

Expiry: 1.0000Y, Tenor: 1.0000Y
  Calibrating to 11 strikes...
  alpha=0.010000, rho=0.000000, nu=0.500000, MSE=1.70e-01

Expiry: 1.0000Y, Tenor: 2.0000Y
  Calibrating to 11 strikes...
  alpha=0.010000, rho=0.000000, nu=0.500000, MSE=1.76e-01

Expiry: 1.0000Y, Tenor: 3.0000Y
  Calibrating to 11 strikes...
  alpha=0.010000, rho=0.000000, nu=0.500000, MSE=1.58e-01

Expiry: 1.0000Y, Tenor: 5.0000Y
  Calibrating to 11 strikes...
  alpha=0.010000, rho=0.000000, nu=0.500000, MSE=1.03e-01

Expiry: 1.0000Y, Tenor: 10.0000Y
  Calibrating to 11 strikes...
  alpha=0.010000, rho=0.000000, nu=0.500000, MSE=7.45e-02

Expiry: 5.0000Y, Tenor: 1.0000Y
  Calibrating to 11 strikes...
  alpha=0.010000, rho=0.000000, nu=0.500000, MSE=1.22e-01

Expiry: 5.0000Y, Tenor: 2.0000Y
  Calibrating to 11 strikes...
  alpha=0.010000, rho=0.000000, nu=0.500000, MSE=1.17e-01

Expiry: 5.0000Y, Tenor: 3.0000Y
  Calibrating to 11 strikes...
  alpha=0.010000, rho=0.000000

,Expiry (Years),Tenor (Years),Num Strikes,Alpha,Beta,Rho,Nu,MSE
0,1.0,1.0,11,0.01,0.75,0.0,0.5,0.169506
1,1.0,2.0,11,0.01,0.75,0.0,0.5,0.176279
2,1.0,3.0,11,0.01,0.75,0.0,0.5,0.158490
3,1.0,5.0,11,0.01,0.75,0.0,0.5,0.103218
4,1.0,10.0,11,0.01,0.75,0.0,0.5,0.074534
5,5.0,1.0,11,0.01,0.75,0.0,0.5,0.122404
6,5.0,2.0,11,0.01,0.75,0.0,0.5,0.117269
7,5.0,3.0,11,0.01,0.75,0.0,0.5,0.110385
8,5.0,5.0,11,0.01,0.75,0.0,0.5,0.079156
9,5.0,10.0,11,0.01,0.75,0.0,0.5,0.062286


In [24]:
# Swaption Pricing Using Calibrated Models

def get_calibrated_sabr_params(T, tenor, df_calib):
    """
    Get calibrated SABR parameters for a given expiry and tenor.
    Uses nearest neighbor if exact match not found.
    
    Parameters:
    -----------
    T : float
        Time to expiry (in years)
    tenor : float
        Swap tenor (in years)
    df_calib : DataFrame
        Calibration results dataframe
        
    Returns:
    --------
    dict : Contains alpha, beta, rho, nu, MSE
    """
    # Find row matching expiry and tenor
    match = df_calib[(df_calib['Expiry (Years)'] == T) & (df_calib['Tenor (Years)'] == tenor)]
    
    if len(match) > 0:
        row = match.iloc[0]
        return {
            'alpha': row['Alpha'],
            'beta': row['Beta'],
            'rho': row['Rho'],
            'nu': row['Nu'],
            'MSE': row['MSE']
        }
    else:
        # Return None if no match
        return None


def get_calibrated_dd_params(T, tenor, df_calib):
    """
    Get calibrated Displaced Diffusion parameters for a given expiry and tenor.
    
    Parameters:
    -----------
    T : float
        Time to expiry (in years)
    tenor : float
        Swap tenor (in years)
    df_calib : DataFrame
        DD calibration results dataframe
        
    Returns:
    --------
    dict : Contains beta, sigma, MSE
    """
    match = df_calib[(df_calib['Expiry (Years)'] == T) & (df_calib['Tenor (Years)'] == tenor)]
    
    if len(match) > 0:
        row = match.iloc[0]
        return {
            'beta': row['Beta'],
            'sigma': row['Sigma'],
            'MSE': row['MSE']
        }
    else:
        return None


def price_swaption_sabr(F, K, T, tenor, df_calib_sabr, payoff_type='payer', annuity=None):
    """
    Price a swaption using calibrated SABR model.
    
    Parameters:
    -----------
    F : float
        Forward swap rate
    K : float
        Strike price
    T : float
        Time to expiry (in years)
    tenor : float
        Swap tenor (in years)
    df_calib_sabr : DataFrame
        SABR calibration results
    payoff_type : str
        'payer' or 'receiver'
    annuity : float
        Annuity factor; if None, uses normalized 1.0
        
    Returns:
    --------
    float : Swaption price
    """
    if annuity is None:
        annuity = 1.0
    
    params = get_calibrated_sabr_params(T, tenor, df_calib_sabr)
    
    if params is None:
        return np.nan
    
    # Get SABR volatility
    sabr_vol = SABR(F, K, T, params['alpha'], params['beta'], params['rho'], params['nu'])
    
    # Price using Black's formula
    r = 0.0
    if payoff_type.lower() == 'payer':
        price = annuity * BlackScholesLognormalPayer(F, K, r, sabr_vol, T)
    elif payoff_type.lower() == 'receiver':
        price = annuity * BlackScholesLognormalReceiver(F, K, r, sabr_vol, T)
    else:
        return np.nan
    
    return price


def price_swaption_dd(F, K, T, tenor, df_calib_dd, payoff_type='payer', annuity=None):
    """
    Price a swaption using calibrated Displaced Diffusion model.
    
    Parameters:
    -----------
    F : float
        Forward swap rate
    K : float
        Strike price
    T : float
        Time to expiry (in years)
    tenor : float
        Swap tenor (in years)
    df_calib_dd : DataFrame
        DD calibration results
    payoff_type : str
        'payer' or 'receiver'
    annuity : float
        Annuity factor; if None, uses normalized 1.0
        
    Returns:
    --------
    float : Swaption price
    """
    if annuity is None:
        annuity = 1.0
    
    params = get_calibrated_dd_params(T, tenor, df_calib_dd)
    
    if params is None:
        return np.nan
    
    beta = params['beta']
    sigma = params['sigma']
    
    # Calculate DD price
    dd_price = DisplacedDiffusion_price(F, K, T, beta, sigma)
    
    return annuity * dd_price


def price_swaption_implied_vol_sabr(F, K, T, tenor, df_calib_sabr):
    """
    Get implied Black volatility for a swaption using calibrated SABR model.
    
    Parameters:
    -----------
    F : float
        Forward swap rate
    K : float
        Strike price
    T : float
        Time to expiry (in years)
    tenor : float
        Swap tenor (in years)
    df_calib_sabr : DataFrame
        SABR calibration results
        
    Returns:
    --------
    float : Implied Black volatility
    """
    params = get_calibrated_sabr_params(T, tenor, df_calib_sabr)
    
    if params is None:
        return np.nan
    
    sabr_vol = SABR(F, K, T, params['alpha'], params['beta'], params['rho'], params['nu'])
    
    return sabr_vol


def price_swaption_implied_vol_dd(F, K, T, tenor, df_calib_dd):
    """
    Get implied Black volatility for a swaption using calibrated DD model.
    
    Parameters:
    -----------
    F : float
        Forward swap rate
    K : float
        Strike price
    T : float
        Time to expiry (in years)
    tenor : float
        Swap tenor (in years)
    df_calib_dd : DataFrame
        DD calibration results
        
    Returns:
    --------
    float : Implied Black volatility
    """
    params = get_calibrated_dd_params(T, tenor, df_calib_dd)
    
    if params is None:
        return np.nan
    
    beta = params['beta']
    sigma = params['sigma']
    
    dd_vol = DisplacedDiffusion_implied_vol(F, K, T, beta, sigma)
    
    return dd_vol


# Price swaptions across a range of strikes
def price_swaption_surface(T, tenor, forward_rate, payoff_type, 
                           strike_pcts=None, model_type='both'):
    """
    Price swaptions across a range of strikes using calibrated model(s).
    
    Parameters:
    -----------
    T : float
        Time to expiry (in years)
    tenor : float
        Swap tenor (in years)
    forward_rate : float
        Forward swap rate
    payoff_type : str
        'payer' or 'receiver'
    strike_pcts : list
        Strike percentages (e.g., [1, 2, 5, 10] for 1%-10% ATM)
    model_type : str
        'sabr', 'dd', or 'both'
        
    Returns:
    --------
    DataFrame : Pricing results
    """
    if strike_pcts is None:
        strike_pcts = np.arange(1, 11)
    
    strikes = forward_rate * strike_pcts / 100.0
    
    results = {'Strike (%)': strike_pcts, 'Strike': strikes}
    
    # Get forward and annuity for this expiry-tenor if available
    annuity_match = df_annuities[(df_annuities['Expiry'] == T) & (df_annuities['Tenor'] == tenor)]
    if len(annuity_match) > 0:
        annuity = annuity_match.iloc[0]['Annuity']
    else:
        annuity = 1.0
    
    if model_type in ['sabr', 'both']:
        sabr_prices = []
        sabr_vols = []
        for strike in strikes:
            price = price_swaption_sabr(forward_rate, strike, T, tenor, df_calibration, payoff_type, annuity)
            vol = price_swaption_implied_vol_sabr(forward_rate, strike, T, tenor, df_calibration)
            sabr_prices.append(price)
            sabr_vols.append(vol)
        results['SABR_Price'] = sabr_prices
        results['SABR_ImpliedVol'] = sabr_vols
    
    if model_type in ['dd', 'both']:
        dd_prices = []
        dd_vols = []
        for strike in strikes:
            price = price_swaption_dd(forward_rate, strike, T, tenor, df_dd_calibration, payoff_type, annuity)
            vol = price_swaption_implied_vol_dd(forward_rate, strike, T, tenor, df_dd_calibration)
            dd_prices.append(price)
            dd_vols.append(vol)
        results['DD_Price'] = dd_prices
        results['DD_ImpliedVol'] = dd_vols
    
    return pd.DataFrame(results)


# ============================================================================
# PRICE SPECIFIC SWAPTIONS
# ============================================================================

strike_percentages = np.arange(1, 11)  # 1% to 10%

print("=" * 100)
print("PRICING SWAPTIONS WITH CALIBRATED MODELS")
print("=" * 100)

# 1. Payer 1x1
print("\n1. PAYER 1x1 SWAPTION - Implied Volatilities and Prices Across Strikes (1%-10%)")
print("-" * 100)
F_1x1 = df_forwards[(df_forwards['Expiry'] == 1.0) & (df_forwards['Tenor'] == 1.0)].iloc[0]['Forward_Rate']
print(f"Forward Rate: {F_1x1:.6f}")

payer_1x1 = price_swaption_surface(T=1.0, tenor=1.0, forward_rate=F_1x1, 
                                   payoff_type='payer', strike_pcts=strike_percentages, model_type='both')
display(payer_1x1.round(6))

# 2. Payer 10x10
print("\n2. PAYER 10x10 SWAPTION - Implied Volatilities and Prices Across Strikes (1%-10%)")
print("-" * 100)
F_10x10 = df_forwards[(df_forwards['Expiry'] == 10.0) & (df_forwards['Tenor'] == 10.0)].iloc[0]['Forward_Rate']
print(f"Forward Rate: {F_10x10:.6f}")

payer_10x10 = price_swaption_surface(T=10.0, tenor=10.0, forward_rate=F_10x10, 
                                     payoff_type='payer', strike_pcts=strike_percentages, model_type='both')
display(payer_10x10.round(6))

# 3. Receiver 30x30 (Note: 30x30 not in calibration, will use 10x10 as proxy or return NaN)
print("\n3. RECEIVER 30x30 SWAPTION - Implied Volatilities and Prices Across Strikes (1%-10%)")
print("-" * 100)
print("Note: 30x30 not in calibration set. Will calculate using long-tenor extrapolation assumptions.")

# For 30x30, use assumptions since not in calibrated set
F_30x30 = calculate_forward_rate(30.0, 30.0)
print(f"Forward Rate (estimated): {F_30x30:.6f}")

# Use 10x10 calibrated params as proxy for longer tenors (conservative approach)
receiver_30x30_sabr = []
receiver_30x30_dd = []
strikes_30x30 = F_30x30 * strike_percentages / 100.0

for strike in strikes_30x30:
    # Use 10x10 params as proxy
    params_sabr = get_calibrated_sabr_params(10.0, 10.0, df_calibration)
    params_dd = get_calibrated_dd_params(10.0, 10.0, df_dd_calibration)
    
    if params_sabr:
        sabr_vol = SABR(F_30x30, strike, 30.0, params_sabr['alpha'], params_sabr['beta'], 
                       params_sabr['rho'], params_sabr['nu'])
        r = 0.0
        annuity_30x30 = calculate_annuity_factor(30.0, 30.0)
        price_sabr = annuity_30x30 * BlackScholesLognormalReceiver(F_30x30, strike, r, sabr_vol, 30.0)
        receiver_30x30_sabr.append({'vol': sabr_vol, 'price': price_sabr})
    else:
        receiver_30x30_sabr.append({'vol': np.nan, 'price': np.nan})
    
    if params_dd:
        dd_vol = DisplacedDiffusion_implied_vol(F_30x30, strike, 30.0, params_dd['beta'], params_dd['sigma'])
        annuity_30x30 = calculate_annuity_factor(30.0, 30.0)
        price_dd = annuity_30x30 * DisplacedDiffusion_price(F_30x30, strike, 30.0, params_dd['beta'], params_dd['sigma'])
        receiver_30x30_dd.append({'vol': dd_vol, 'price': price_dd})
    else:
        receiver_30x30_dd.append({'vol': np.nan, 'price': np.nan})

receiver_30x30 = pd.DataFrame({
    'Strike (%)': strike_percentages,
    'Strike': strikes_30x30,
    'SABR_ImpliedVol': [x['vol'] for x in receiver_30x30_sabr],
    'SABR_Price': [x['price'] for x in receiver_30x30_sabr],
    'DD_ImpliedVol': [x['vol'] for x in receiver_30x30_dd],
    'DD_Price': [x['price'] for x in receiver_30x30_dd]
})
display(receiver_30x30.round(6))

print("\n" + "=" * 100)


PRICING SWAPTIONS WITH CALIBRATED MODELS

1. PAYER 1x1 SWAPTION - Implied Volatilities and Prices Across Strikes (1%-10%)
----------------------------------------------------------------------------------------------------
Forward Rate: 0.022079


,Strike (%),Strike,SABR_Price,SABR_ImpliedVol,DD_Price,DD_ImpliedVol
0,1,0.000221,0.021160,0.483459,0.021160,1.050543
1,2,0.000442,0.020946,0.423975,0.020946,0.896908
2,3,0.000662,0.020733,0.388011,0.020733,0.811818
3,4,0.000883,0.020519,0.361976,0.020519,0.753872
4,5,0.001104,0.020305,0.341485,0.020305,0.710444
5,6,0.001325,0.020091,0.324546,0.020091,0.676016
6,7,0.001546,0.019878,0.310085,0.019878,0.647691
7,8,0.001766,0.019664,0.297452,0.019664,0.623763
8,9,0.001987,0.019450,0.286224,0.019450,0.603143
9,10,0.002208,0.019236,0.276112,0.019236,0.585097



2. PAYER 10x10 SWAPTION - Implied Volatilities and Prices Across Strikes (1%-10%)
----------------------------------------------------------------------------------------------------
Forward Rate: 0.027194


,Strike (%),Strike,SABR_Price,SABR_ImpliedVol,DD_Price,DD_ImpliedVol
0,1,0.000272,0.199316,0.565877,0.203735,NaN
1,2,0.000544,0.197333,0.496156,0.201956,NaN
2,3,0.000816,0.195346,0.454000,0.200186,1.288303
3,4,0.001088,0.193357,0.423482,0.198424,1.046085
4,5,0.001360,0.191366,0.399462,0.196671,0.925686
5,6,0.001632,0.189375,0.379606,0.194926,0.847164
6,7,0.001904,0.187383,0.362654,0.193190,0.789909
7,8,0.002175,0.185390,0.347845,0.191463,0.745435
8,9,0.002447,0.183396,0.334684,0.189745,0.709435
9,10,0.002719,0.181403,0.322830,0.188036,0.679429



3. RECEIVER 30x30 SWAPTION - Implied Volatilities and Prices Across Strikes (1%-10%)
----------------------------------------------------------------------------------------------------
Note: 30x30 not in calibration set. Will calculate using long-tenor extrapolation assumptions.
Forward Rate (estimated): 0.030302


,Strike (%),Strike,SABR_ImpliedVol,SABR_Price,DD_ImpliedVol,DD_Price
0,1,0.000303,0.756645,0.002840,NaN,0.417722
1,2,0.000606,0.663352,0.005052,NaN,0.415249
2,3,0.000909,0.606941,0.006943,NaN,0.412792
3,4,0.001212,0.566104,0.008611,NaN,0.410352
4,5,0.001515,0.533961,0.010111,NaN,0.407928
5,6,0.001818,0.507391,0.011475,NaN,0.405520
6,7,0.002121,0.484707,0.012728,NaN,0.403128
7,8,0.002424,0.464890,0.013886,NaN,0.400752
8,9,0.002727,0.447278,0.014961,NaN,0.398392
9,10,0.003030,0.431417,0.015963,NaN,0.396048
